# SPECTRA Rubin RSP Data Release Test

This notebook is meant to be run directly on the Rubin Science Platform. It tests the Rubin data-release path currently supported by SPECTRA: the RSP DP0.2/DC2 object catalog, `dp02_dc2_catalogs.Object`.

The workflow starts from the curated `example_configs/config_dp02_test.yaml`, writes a notebook-local copy with a small object cap, runs the pipeline on RSP-accessible Rubin data, and inspects the resulting `fit_summary.csv`.

## 1. Set Up the Repository Path

Run this notebook from the repository root or from the `notebooks/` directory.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if not (repo_root / "src").exists():
    raise RuntimeError("Run this notebook from the SPECTRA repository root or from notebooks/.")

sys.path.insert(0, str(repo_root))
repo_root

## 2. Imports and RSP Authentication

When this notebook runs on the Rubin Science Platform, it reuses the platform-provided token from the notebook environment. The manual prompt is only a fallback for running the notebook somewhere else.

In [ ]:
import getpass
import yaml
import pandas as pd

from src.cli import validate_config
from src.main import main
from src.data.rubin_query import RubinDataQuery


def configure_rsp_token():
    for env_name in ("RSP_TOKEN", "ACCESS_TOKEN", "JUPYTERHUB_API_TOKEN"):
        token = os.environ.get(env_name)
        if token:
            os.environ["RSP_TOKEN"] = token
            return env_name

    try:
        from rubin_jupyter_utils.lab.notebook import utils as rsp_utils

        token = rsp_utils.get_access_token()
        if token:
            os.environ["RSP_TOKEN"] = token
            return "rubin_jupyter_utils"
    except Exception:
        pass

    os.environ["RSP_TOKEN"] = getpass.getpass("Rubin RSP token: ")
    return "manual prompt"


auth_source = configure_rsp_token()
print(f"RSP authentication configured from: {auth_source}")

## 3. Prepare a Small RSP Data-Release Config

This cell copies the public DP0.2 example config and makes the run intentionally small. The `rubin_catalog` variable is the RSP table selector; keep the default for the currently supported DP0.2/DC2 catalog. The TAP URL is taken from the active RSP environment when available.

In [ ]:
base_config_path = repo_root / "example_configs" / "config_dp02_test.yaml"
with base_config_path.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

rubin_release_label = "Rubin RSP DP0.2/DC2"
rubin_catalog = "dp02_dc2_catalogs.Object"
rsp_base_url = os.environ.get("EXTERNAL_INSTANCE_URL", "https://data.lsst.cloud").rstrip("/")
rubin_tap_url = f"{rsp_base_url}/api/tap"

config["input"]["max_objects"] = 2
config["input"]["radius_arcsec"] = 30.0
config["rubin"]["catalog"] = rubin_catalog
config["rubin"]["tap_url"] = rubin_tap_url
config["plotting"]["output_dir"] = str(repo_root / "outputs" / "notebook_rsp_release_test")
config["plotting"]["show_plots"] = False
config["fitting"]["method"] = "ml"

notebook_config_path = repo_root / "rsp_config.yaml"
notebook_config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print(f"Testing {rubin_release_label} catalog: {rubin_catalog}")
print(f"TAP URL: {rubin_tap_url}")
print(notebook_config_path)
config

## 4. Validate the Config

This catches missing required sections before sending a query to Rubin.

In [ ]:
status = validate_config(str(notebook_config_path))
if status != 0:
    raise RuntimeError(f"Config validation failed with status {status}")

print("Config validation passed.")

## 5. Preview the RSP Rubin Query

This quick preview checks that RSP authentication, the selected catalog, and the cone search work before running the fitter.

In [ ]:
query = RubinDataQuery(config=config)
preview = query.cone_search(
    ra=config["input"]["ra"],
    dec=config["input"]["dec"],
    radius_arcsec=config["input"]["radius_arcsec"],
    catalog=config["rubin"]["catalog"],
    flux_type=config["rubin"].get("flux_type", "cModelFlux"),
    bands=config["rubin"].get("bands"),
    max_objects=config["input"]["max_objects"],
)

print(f"Loaded {len(preview)} object(s) from {config['rubin']['catalog']}.")
for object_id, phot_data in preview:
    print(object_id, phot_data.get("bands"), phot_data["obs_flux"].shape)

## 6. Run SPECTRA

This runs maximum-likelihood fitting on the same small RSP Rubin selection and writes plots plus a summary table under `outputs/notebook_rsp_release_test/`.

In [ ]:
main(str(notebook_config_path))

## 7. Inspect Fit Summary

The table below is the first pass quality check. Pay closest attention to `chi2_red`, parameter values at prior boundaries, and whether any object produced missing values.

In [ ]:
summary_path = Path(config["plotting"]["output_dir"]) / "fit_summary.csv"
summary = pd.read_csv(summary_path)
summary

## 8. Quick Diagnostics

This cell flags high reduced chi-square values and lists the per-object output folders.

In [ ]:
display_cols = [col for col in ["object_id", "redshift", "chi2_red", "mass", "age", "metallicity", "dust"] if col in summary.columns]
display(summary[display_cols])

poor = summary.loc[summary["chi2_red"] > 2, display_cols]
if len(poor):
    print("Objects with chi2_red > 2; inspect residual plots before using these scientifically:")
    display(poor)
else:
    print("All objects have chi2_red <= 2 in this smoke test.")

for path in sorted(Path(config["plotting"]["output_dir"]).glob("*")):
    if path.is_dir():
        print(path.relative_to(repo_root))